# STEPR0 — Aanstelling cue distribution over time

Cue inventory for **Track R** (*vergadering-aanstelling*): formula families in [`formulation/aanstelling_cues.py`](../formulation/aanstelling_cues.py) (*creditief*, *sessie genomen*, *verwelkom*, *aengenaem*, *op den heer*, *holland_multi*).

- Corpus: `resolutions_flat` via `data_io`
- Gold overlay: `aanstelling_labelled` positives (213 ids); R1 selection frozen in `aanstelling_selected`
- Dates: ISO strings → year/month by slice (no `pd.to_datetime`; corpus includes pre-1678)
- Heavy cell loads full text once; re-run only when cues change
- **`record_type`**: REPUBLIC parquet `type` (this slice is all `resolution` today)
- **`resolution_type`**: derived class — `aanstelling_gold` → `missive_receipt` → `aanstelling_cue` → `missive_open` → `other` (first match wins)


In [ ]:
import re

import matplotlib.pyplot as plt
import pandas as pd
from data_io import load, resolve

from formulation.aanstelling_cues import CUE_PATTERNS, add_aanstelling_cues

# Track L overlap (same STEPL0 cues as stepl0_missive_cue_time.ipynb)
MISSIVE_CLOSE = re.compile(
    r"geen\s+resolutie\s+(?:is\s+)?gevallen|"
    r"geen\s+resolutie\s+vere[iy]sch",
    re.I,
)
MISSIVE_OPEN = re.compile(
    r"ontfangen\s+een(?:en)?\s+(?:missive|missiue|brief|brieff)",
    re.I,
)

labelled = load("aanstelling_labelled")
gold_ids = set(labelled.loc[labelled["label"] == "aanstelling", "id"])
label_by_id = labelled.set_index("id")["label"]
relocations = load("aanstelling_gold_relocations")
print("gold ids", len(gold_ids))
print("relocations", len(relocations))
print("cue families", list(CUE_PATTERNS))
print("flat", resolve("resolutions_flat"))


## Score cues (once)

Loads `id`, `date`, `resolutions_text` only. Expect ~10–60s depending on disk.


In [ ]:
df = pd.read_parquet(
    resolve("resolutions_flat"),
    columns=["id", "type", "date", "resolutions_text"],
)
date_s = df["date"].astype(str)
df = df.assign(
    year=pd.to_numeric(date_s.str[:4], errors="coerce"),
    month=pd.to_numeric(date_s.str[5:7], errors="coerce"),
)
text = df["resolutions_text"].fillna("").astype(str)
df = df.assign(
    missive_open=text.str.contains(MISSIVE_OPEN, regex=True),
    missive_close=text.str.contains(MISSIVE_CLOSE, regex=True),
)
df = add_aanstelling_cues(df)
df = df.assign(
    record_type=df["type"].astype(str),
    label=df["id"].map(label_by_id).fillna("unlabelled"),
    is_gold=df["id"].isin(gold_ids),
)
resolution_type = pd.Series("other", index=df.index, dtype="string")
resolution_type = resolution_type.mask(df["missive_open"], "missive_open")
resolution_type = resolution_type.mask(df["has_aanstelling_cue"], "aanstelling_cue")
resolution_type = resolution_type.mask(
    df["missive_open"] & df["missive_close"], "missive_receipt"
)
resolution_type = resolution_type.mask(df["label"].eq("aanstelling"), "aanstelling_gold")
df = df.assign(resolution_type=resolution_type)

cue_cols = [f"cue_{name}" for name in CUE_PATTERNS]
cue = df[
    [
        "id",
        "record_type",
        "resolution_type",
        "label",
        "year",
        "month",
        "is_gold",
        "has_aanstelling_cue",
        "missive_open",
        "missive_close",
        "cue_hits",
        *cue_cols,
    ]
].copy()
cue = cue.assign(gold_cue=cue["is_gold"] & cue["has_aanstelling_cue"])

n_gold_in_flat = int(cue["is_gold"].sum())
print(f"n={len(cue):,} with_cue={cue['has_aanstelling_cue'].sum():,} gold_in_flat={n_gold_in_flat}")
print("year range", int(cue["year"].min()), "–", int(cue["year"].max()))
print(
    "gold cue rate",
    round(float(cue.loc[cue["is_gold"], "has_aanstelling_cue"].mean()), 4),
)
print("record_type:", cue["record_type"].value_counts().to_dict())
print("resolution_type:")
print(cue["resolution_type"].value_counts())


## By year

Absolute counts and share of resolutions that year (`has_aanstelling_cue / n`). Gold positives in flat overlaid where present.


In [ ]:
by_year = (
    cue.groupby("year", dropna=True)
    .agg(
        n=("id", "size"),
        with_cue=("has_aanstelling_cue", "sum"),
        gold=("is_gold", "sum"),
        gold_with_cue=("gold_cue", "sum"),
    )
    .astype({"with_cue": int, "gold": int, "gold_with_cue": int})
    .assign(cue_share=lambda d: d["with_cue"] / d["n"])
)
by_year.tail(10)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
by_year[["with_cue", "gold"]].plot(ax=axes[0], lw=1)
axes[0].set_ylabel("count")
axes[0].set_title("Aanstelling cues by year")
axes[0].legend(loc="upper left")
by_year["cue_share"].plot(ax=axes[1], color="C2", lw=1)
axes[1].set_ylabel("with_cue / n")
axes[1].set_xlabel("year")
axes[1].set_title("Share of resolutions with any aanstelling cue")
fig.tight_layout()


## Cue families by year (1705–1795 window)

Which formula families drive volume in the delegate gold window.


In [ ]:
window = cue.loc[(cue["year"] >= 1705) & (cue["year"] <= 1795)]
fam = {name: int(window[f"cue_{name}"].sum()) for name in CUE_PATTERNS}
fam_series = pd.Series(fam).sort_values(ascending=False)
fam_series


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
fam_series.plot.bar(ax=ax, rot=45)
ax.set_title("Cue-family hits (1705–1795)")
ax.set_ylabel("count")
plt.setp(ax.get_xticklabels(), ha="right")
fig.tight_layout()


## By month (all years pooled)

Seasonality of `has_aanstelling_cue` vs baseline volume.


In [ ]:
by_month = (
    cue.dropna(subset=["month"])
    .groupby("month")
    .agg(n=("id", "size"), with_cue=("has_aanstelling_cue", "sum"))
    .astype(int)
    .assign(cue_share=lambda d: d["with_cue"] / d["n"])
)
by_month


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
by_month[["n", "with_cue"]].plot.bar(ax=axes[0], rot=0)
axes[0].set_title("Volume vs with_cue (all years)")
axes[0].set_xlabel("month")
by_month["cue_share"].plot.bar(ax=axes[1], color="C2", rot=0)
axes[1].set_title("with_cue / n by month")
axes[1].set_xlabel("month")
fig.tight_layout()


## One year detail + year×month heatmap

Default focal year `1780`. Change `FOCAL_YEAR` as needed.


In [ ]:
FOCAL_YEAR = 1780

focal = cue.loc[cue["year"] == FOCAL_YEAR]
focal_month = (
    focal.groupby("month")
    .agg(
        n=("id", "size"),
        with_cue=("has_aanstelling_cue", "sum"),
        gold=("is_gold", "sum"),
    )
    .astype(int)
)
print(f"year {FOCAL_YEAR}")
display(focal_month)

heat = (
    cue.dropna(subset=["year", "month"])
    .loc[lambda d: (d["year"] >= 1700) & (d["year"] <= 1795)]
    .pivot_table(
        index="year",
        columns="month",
        values="has_aanstelling_cue",
        aggfunc="sum",
        fill_value=0,
    )
)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(heat.to_numpy(), aspect="auto", cmap="YlOrRd", interpolation="nearest")
ax.set_yticks(range(0, len(heat), 5))
ax.set_yticklabels(heat.index[::5].astype(int))
ax.set_xticks(range(12))
ax.set_xticklabels(range(1, 13))
ax.set_xlabel("month")
ax.set_ylabel("year")
ax.set_title("aanstelling-cue counts (1700–1795)")
fig.colorbar(im, ax=ax, shrink=0.6, label="with_cue")
fig.tight_layout()


## Spot-check samples

Gold positives vs high-recall cue-only rows in the focal year (sanity: bare *verwelkom* / referrals without appointment frames).


In [ ]:
FOCAL_YEAR  # from earlier cell


def _sample_ids(mask, n=5):
    return cue.loc[mask, "id"].head(n).tolist()


cue_only = _sample_ids(
    (cue["year"] == FOCAL_YEAR) & cue["has_aanstelling_cue"] & ~cue["is_gold"]
)
gold_ids_sample = _sample_ids((cue["year"] == FOCAL_YEAR) & cue["is_gold"])
print("cue-only (not in labelled gold)", cue_only)
print("labelled gold in year", gold_ids_sample)

want = set(cue_only + gold_ids_sample)
peek = df.loc[df["id"].isin(want), ["id", "cue_hits", "resolutions_text"]]
for _, row in peek.iterrows():
    tag = "gold" if row["id"] in gold_ids_sample else "cue-only"
    print(f"\n--- {tag} hits={row['cue_hits']} {row['id']} ---")
    print(str(row["resolutions_text"])[:400])
